In [ ]:
import numpy as np
import cv2
from PIL import Image
from pathlib import Path

In [ ]:
DATA_DIR = Path.cwd() / "data"
ORIGINAL_DIR = DATA_DIR / "original"
LABELS_DIR = DATA_DIR / "labels"

In [ ]:
def list_image_pairs():
    originals = sorted(ORIGINAL_DIR.glob("*.ppm"))
    labels = sorted(LABELS_DIR.glob("*.vk.ppm"))
    pairs = []
    for orig in originals:
        lbl = LABELS_DIR / f"{orig.stem}.vk.ppm"
        if lbl.exists():
            pairs.append((orig, lbl))
    return pairs


pairs = list_image_pairs()
print(f"Found {len(pairs)} input-target pairs")
for inp, tgt in pairs[:3]:
    print(f"  {inp.name} -> {tgt.name}")

In [ ]:
def extract_rgb_channels(image_path: Path):
    img = np.array(Image.open(image_path))
    return img[:, :, 0], img[:, :, 1], img[:, :, 2]

In [ ]:
def create_eye_mask(image_path: Path):
    img = cv2.imread(str(image_path))
    mask = np.zeros(img.shape[:2], np.uint8)
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    h, w = img.shape[:2]
    margin = 5
    rect = (margin, margin, w - 2 * margin, h - 2 * margin)
    cv2.grabCut(img, mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)
    result = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0).astype(np.uint8)
    return result > 0

In [ ]:
def show_images(images, titles=None, cols = None, figsize_per_image=4, cmap="gray"):
    n = len(images)
    if cols is None:
        cols = n

    rows = max(1, (n + cols - 1) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * figsize_per_image, rows * figsize_per_image))
    axes = np.atleast_1d(axes).ravel()
    for i, src in enumerate(images):
        if isinstance(src, (str, Path)):
            img = np.array(Image.open(src))
        else:
            img = src
        axes[i].imshow(img, cmap=cmap)
        if titles and i < len(titles):
            axes[i].set_title(titles[i])
        axes[i].axis('off')
    for j in range(n, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
for orig, targ in pairs[:4]:
    r, g, b = extract_rgb_channels(orig)
    mask = create_eye_mask(orig)
    show_images([orig, targ, r, g, mask])